# Regression in practice: practice notebook

7 exercises, each with hidden tests. Data is loaded as **polars**; pandas only appears where a library insists on it.

**How to use it**
1. Run the three setup cells below once (Runtime → Run all works too; the exercise cells just print `Wrong answer` until you solve them).
2. For each exercise, write your code in the solution cell, run it, then run the `check(...)` cell under it. It prints **Accepted** or which test failed and why.
3. Stuck? `solution("r1")` prints the reference solution.

Opened from the roadmap it runs in Google Colab. To keep your work, use File → Save a copy in Drive the first time.

In [ ]:
# Install anything missing (Colab already has numpy, scipy, statsmodels, scikit-learn, matplotlib)
import importlib.util, subprocess, sys
for _p in ['polars']:
    if importlib.util.find_spec(_p) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _p], check=False)


## Setup: data
The data every exercise uses. Read it if you like; don't change it.

In [ ]:
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

def rdata(name, package):
    """Load an R dataset (via statsmodels / Rdatasets) as a polars DataFrame, NaN → null."""
    df = pl.from_pandas(sm.datasets.get_rdataset(name, package, cache=True).data.reset_index(drop=True)).drop("rownames", strict=False)
    return df.with_columns(pl.col(pl.Float64).fill_nan(None))

airquality = rdata("airquality", "datasets")
auto = rdata("Auto", "ISLR")
cats = rdata("cats", "MASS")

In [ ]:
# Test runner: check(id) and solution(id)
import base64 as _b64, json as _json, io as _io, contextlib as _ctx, traceback as _tb
_LAB = _json.loads(_b64.b64decode("eyJyMSI6IHsic29sdXRpb24iOiAiZGVmIHNpbV9zbG9wZXMoZXJyb3JzPVwibm9ybWFsXCIsIG49MjAwLCByZXBzPTEwMDAsIHNlZWQ9MCk6XG4gICAgcm5nID0gbnAucmFuZG9tLmRlZmF1bHRfcm5nKHNlZWQpXG4gICAgb3V0ID0gbnAuZW1wdHkocmVwcylcbiAgICBmb3IgaSBpbiByYW5nZShyZXBzKTpcbiAgICAgICAgeCA9IHJuZy51bmlmb3JtKC0zLCAzLCBuKVxuICAgICAgICBlID0gcm5nLnN0YW5kYXJkX2NhdWNoeShuKSBpZiBlcnJvcnMgPT0gXCJjYXVjaHlcIiBlbHNlIHJuZy5ub3JtYWwoc2l6ZT1uKVxuICAgICAgICB5ID0gNSAtIDIgKiB4ICsgZVxuICAgICAgICBvdXRbaV0gPSBzbS5PTFMoeSwgc20uYWRkX2NvbnN0YW50KHgpKS5maXQoKS5wYXJhbXNbMV1cbiAgICByZXR1cm4gb3V0XG5cbmRlZiBzaW1fYXR0ZW51YXRpb24obj0xMDAsIHJlcHM9MTAwMCwgc2VlZD0wKTpcbiAgICBybmcgPSBucC5yYW5kb20uZGVmYXVsdF9ybmcoc2VlZClcbiAgICBvdXQgPSBucC5lbXB0eShyZXBzKVxuICAgIGZvciBpIGluIHJhbmdlKHJlcHMpOlxuICAgICAgICB4ID0gcm5nLnVuaWZvcm0oMCwgMSwgbilcbiAgICAgICAgdyA9IHggKyBybmcubm9ybWFsKDAsIG5wLnNxcnQoMiksIG4pXG4gICAgICAgIHkgPSA1ICsgMyAqIHggKyBybmcubm9ybWFsKHNpemU9bilcbiAgICAgICAgb3V0W2ldID0gc20uT0xTKHksIHNtLmFkZF9jb25zdGFudCh3KSkuZml0KCkucGFyYW1zWzFdXG4gICAgcmV0dXJuIG91dFxuIiwgInRlc3RzIjogIlxuZGVmIGNsb3NlKGEsIGIsIHJlbD0xZS02LCB3aGF0PVwidmFsdWVcIik6XG4gICAgYXNzZXJ0IGEgaXMgbm90IE5vbmUsIFwieW91ciBmdW5jdGlvbiByZXR1cm5lZCBOb25lLiBEaWQgeW91IGZvcmdldCB0byByZXR1cm4/XCJcbiAgICBhID0gZmxvYXQoYSk7IGIgPSBmbG9hdChiKVxuICAgIGFzc2VydCBhYnMoYSAtIGIpIDw9IHJlbCAqIG1heCgxLjAsIGFicyhiKSksIGZcInt3aGF0fSA9IHthOi42Z30sIGV4cGVjdGVkIHtiOi42Z31cIlxuZGVmIGJldHdlZW4oYSwgbG8sIGhpLCB3aGF0PVwidmFsdWVcIik6XG4gICAgYXNzZXJ0IGEgaXMgbm90IE5vbmUsIFwieW91ciBmdW5jdGlvbiByZXR1cm5lZCBOb25lLiBEaWQgeW91IGZvcmdldCB0byByZXR1cm4/XCJcbiAgICBhc3NlcnQgbG8gPD0gZmxvYXQoYSkgPD0gaGksIGZcInt3aGF0fSA9IHtmbG9hdChhKTouNGZ9LCBleHBlY3RlZCBiZXR3ZWVuIHtsb30gYW5kIHtoaX1cIlxuZGVmIHNhbWVfa2V5cyhhLCBiLCB3aGF0PVwicmVzdWx0XCIpOlxuICAgIGFzc2VydCBpc2luc3RhbmNlKGEsIGRpY3QpLCBmXCJ7d2hhdH0gc2hvdWxkIGJlIGEgZGljdCwgZ290IHt0eXBlKGEpLl9fbmFtZV9ffVwiXG4gICAgYXNzZXJ0IHNldChhKSA9PSBzZXQoYiksIGZcInt3aGF0fSBrZXlzIHtzb3J0ZWQoYSl9LCBleHBlY3RlZCB7c29ydGVkKGIpfVwiXG5cbmRlZiBfbm9ybWFsKCk6XG4gICAgcyA9IG5wLmFzYXJyYXkoVVtcInNpbV9zbG9wZXNcIl0oXCJub3JtYWxcIikpXG4gICAgYXNzZXJ0IHMuc2hhcGUgPT0gKDEwMDAsKSwgZlwic2hhcGUge3Muc2hhcGV9LCBleHBlY3RlZCAoMTAwMCwpXCJcbiAgICBiZXR3ZWVuKHMubWVhbigpLCAtMi4wMSwgLTEuOTksIFwibWVhbiBzbG9wZSAoTm9ybWFsIGVycm9ycylcIilcbiAgICBiZXR3ZWVuKHMuc3RkKCksIDAuMDM0LCAwLjA0OCwgXCJzdGQgb2Ygc2xvcGVzIChOb3JtYWwgZXJyb3JzKVwiKVxuZGVmIF9jYXVjaHkoKTpcbiAgICBzID0gbnAuYXNhcnJheShVW1wic2ltX3Nsb3Blc1wiXShcImNhdWNoeVwiLCBzZWVkPTEpKTsgbiA9IG5wLmFzYXJyYXkoVVtcInNpbV9zbG9wZXNcIl0oXCJub3JtYWxcIiwgc2VlZD0xKSlcbiAgICBiZXR3ZWVuKG5wLm1lZGlhbihzKSwgLTIuMTUsIC0xLjg1LCBcIm1lZGlhbiBzbG9wZSAoQ2F1Y2h5IGVycm9ycylcIilcbiAgICBzcHJlYWQgPSBsYW1iZGEgdjogbnAucGVyY2VudGlsZSh2LCA5OSkgLSBucC5wZXJjZW50aWxlKHYsIDEpXG4gICAgYXNzZXJ0IHNwcmVhZChzKSA+IDUgKiBzcHJlYWQobiksIFwiQ2F1Y2h5IHNsb3BlcyBzaG91bGQgaGF2ZSBmYXIgaGVhdmllciB0YWlscyB0aGFuIE5vcm1hbCBvbmVzXCJcbmRlZiBfYXR0KCk6XG4gICAgdyA9IG5wLmFzYXJyYXkoVVtcInNpbV9hdHRlbnVhdGlvblwiXSgpKVxuICAgIGJldHdlZW4ody5tZWFuKCksIDAuMDMsIDAuMjUsIFwibWVhbiBzbG9wZSB3aXRoIGVycm9ycyBpbiBYXCIpXG5URVNUUyA9IFsoXCJOb3JtYWwgZXJyb3JzXCIsIF9ub3JtYWwpLCAoXCJDYXVjaHkgZXJyb3JzXCIsIF9jYXVjaHkpLCAoXCJlcnJvcnMgaW4gWFwiLCBfYXR0KV1cbiIsICJsYW5nIjogInB5In0sICJyMiI6IHsic29sdXRpb24iOiAiZGVmIG96b25lX2ZpdChkZik6XG4gICAgZCA9IGRmLmRyb3BfbnVsbHMoW1wiT3pvbmVcIiwgXCJTb2xhci5SXCJdKVxuICAgIHggPSBkW1wiU29sYXIuUlwiXS50b19udW1weSgpLmFzdHlwZShmbG9hdClcbiAgICB5ID0gZFtcIk96b25lXCJdLnRvX251bXB5KCkuYXN0eXBlKGZsb2F0KVxuICAgIG0gPSBzbS5PTFMoeSwgc20uYWRkX2NvbnN0YW50KHgpKS5maXQoKVxuICAgIGxvLCBoaSA9IG0uY29uZl9pbnQoYWxwaGE9MC4xMClbMV1cbiAgICByZXR1cm4ge1wiaW50ZXJjZXB0XCI6IGZsb2F0KG0ucGFyYW1zWzBdKSwgXCJzbG9wZVwiOiBmbG9hdChtLnBhcmFtc1sxXSksXG4gICAgICAgICAgICBcImNpOTBcIjogKGZsb2F0KGxvKSwgZmxvYXQoaGkpKSwgXCJwX3ZhbHVlXCI6IGZsb2F0KG0ucHZhbHVlc1sxXSl9XG4iLCAidGVzdHMiOiAiXG5kZWYgY2xvc2UoYSwgYiwgcmVsPTFlLTYsIHdoYXQ9XCJ2YWx1ZVwiKTpcbiAgICBhc3NlcnQgYSBpcyBub3QgTm9uZSwgXCJ5b3VyIGZ1bmN0aW9uIHJldHVybmVkIE5vbmUuIERpZCB5b3UgZm9yZ2V0IHRvIHJldHVybj9cIlxuICAgIGEgPSBmbG9hdChhKTsgYiA9IGZsb2F0KGIpXG4gICAgYXNzZXJ0IGFicyhhIC0gYikgPD0gcmVsICogbWF4KDEuMCwgYWJzKGIpKSwgZlwie3doYXR9ID0ge2E6LjZnfSwgZXhwZWN0ZWQge2I6LjZnfVwiXG5kZWYgYmV0d2VlbihhLCBsbywgaGksIHdoYXQ9XCJ2YWx1ZVwiKTpcbiAgICBhc3NlcnQgYSBpcyBub3QgTm9uZSwgXCJ5b3VyIGZ1bmN0aW9uIHJldHVybmVkIE5vbmUuIERpZCB5b3UgZm9yZ2V0IHRvIHJldHVybj9cIlxuICAgIGFzc2VydCBsbyA8PSBmbG9hdChhKSA8PSBoaSwgZlwie3doYXR9ID0ge2Zsb2F0KGEpOi40Zn0sIGV4cGVjdGVkIGJldHdlZW4ge2xvfSBhbmQge2hpfVwiXG5kZWYgc2FtZV9rZXlzKGEsIGIsIHdoYXQ9XCJyZXN1bHRcIik6XG4gICAgYXNzZXJ0IGlzaW5zdGFuY2UoYSwgZGljdCksIGZcInt3aGF0fSBzaG91bGQgYmUgYSBkaWN0LCBnb3Qge3R5cGUoYSkuX19uYW1lX199XCJcbiAgICBhc3NlcnQgc2V0KGEpID09IHNldChiKSwgZlwie3doYXR9IGtleXMge3NvcnRlZChhKX0sIGV4cGVjdGVkIHtzb3J0ZWQoYil9XCJcblxuZGVmIF90KGRmKTpcbiAgICBhID0gVVtcIm96b25lX2ZpdFwiXShkZik7IGIgPSBSW1wib3pvbmVfZml0XCJdKGRmKTsgc2FtZV9rZXlzKGEsIGIpXG4gICAgZm9yIGsgaW4gKFwiaW50ZXJjZXB0XCIsIFwic2xvcGVcIiwgXCJwX3ZhbHVlXCIpOiBjbG9zZShhW2tdLCBiW2tdLCAxZS02LCBrKVxuICAgIGNsb3NlKGFbXCJjaTkwXCJdWzBdLCBiW1wiY2k5MFwiXVswXSwgMWUtNiwgXCJjaTkwIGxvd1wiKTsgY2xvc2UoYVtcImNpOTBcIl1bMV0sIGJbXCJjaTkwXCJdWzFdLCAxZS02LCBcImNpOTAgaGlnaFwiKVxuVEVTVFMgPSBbKFwiYWlycXVhbGl0eVwiLCBsYW1iZGE6IF90KGFpcnF1YWxpdHkpKSwgKFwiaGlkZGVuOiBzdW1tZXIgbW9udGhzIG9ubHlcIiwgbGFtYmRhOiBfdChhaXJxdWFsaXR5LmZpbHRlcihwbC5jb2woXCJNb250aFwiKSA+PSA2KSkpXVxuIiwgImxhbmciOiAicHkifSwgInIzIjogeyJzb2x1dGlvbiI6ICJLSU5EUyA9IHtcImxpbmVhclwiOiBwbC5jb2woXCJ3ZWlnaHRcIiksIFwic3FydFwiOiBwbC5jb2woXCJ3ZWlnaHRcIikuc3FydCgpLFxuICAgICAgICAgXCJsb2dcIjogcGwuY29sKFwid2VpZ2h0XCIpLmxvZygpLCBcImludmVyc2VcIjogMSAvIHBsLmNvbChcIndlaWdodFwiKX1cblxuZGVmIF9maXQoZGYsIGtpbmQpOlxuICAgIHggPSBkZi5zZWxlY3QoS0lORFNba2luZF0uY2FzdChwbC5GbG9hdDY0KSkudG9fc2VyaWVzKCkudG9fbnVtcHkoKVxuICAgIHJldHVybiBzbS5PTFMoZGZbXCJtcGdcIl0udG9fbnVtcHkoKS5hc3R5cGUoZmxvYXQpLCBzbS5hZGRfY29uc3RhbnQoeCkpLmZpdCgpXG5cbmRlZiB0cmFuc2Zvcm1fcjIoZGYpOlxuICAgIHJldHVybiB7azogZmxvYXQoX2ZpdChkZiwgaykucnNxdWFyZWQpIGZvciBrIGluIEtJTkRTfVxuXG5kZWYgc2xvcGVfY2k4MChkZiwga2luZCk6XG4gICAgbG8sIGhpID0gX2ZpdChkZiwga2luZCkuY29uZl9pbnQoYWxwaGE9MC4yMClbMV1cbiAgICByZXR1cm4gZmxvYXQobG8pLCBmbG9hdChoaSlcbiIsICJ0ZXN0cyI6ICJcbmRlZiBjbG9zZShhLCBiLCByZWw9MWUtNiwgd2hhdD1cInZhbHVlXCIpOlxuICAgIGFzc2VydCBhIGlzIG5vdCBOb25lLCBcInlvdXIgZnVuY3Rpb24gcmV0dXJuZWQgTm9uZS4gRGlkIHlvdSBmb3JnZXQgdG8gcmV0dXJuP1wiXG4gICAgYSA9IGZsb2F0KGEpOyBiID0gZmxvYXQoYilcbiAgICBhc3NlcnQgYWJzKGEgLSBiKSA8PSByZWwgKiBtYXgoMS4wLCBhYnMoYikpLCBmXCJ7d2hhdH0gPSB7YTouNmd9LCBleHBlY3RlZCB7YjouNmd9XCJcbmRlZiBiZXR3ZWVuKGEsIGxvLCBoaSwgd2hhdD1cInZhbHVlXCIpOlxuICAgIGFzc2VydCBhIGlzIG5vdCBOb25lLCBcInlvdXIgZnVuY3Rpb24gcmV0dXJuZWQgTm9uZS4gRGlkIHlvdSBmb3JnZXQgdG8gcmV0dXJuP1wiXG4gICAgYXNzZXJ0IGxvIDw9IGZsb2F0KGEpIDw9IGhpLCBmXCJ7d2hhdH0gPSB7ZmxvYXQoYSk6LjRmfSwgZXhwZWN0ZWQgYmV0d2VlbiB7bG99IGFuZCB7aGl9XCJcbmRlZiBzYW1lX2tleXMoYSwgYiwgd2hhdD1cInJlc3VsdFwiKTpcbiAgICBhc3NlcnQgaXNpbnN0YW5jZShhLCBkaWN0KSwgZlwie3doYXR9IHNob3VsZCBiZSBhIGRpY3QsIGdvdCB7dHlwZShhKS5fX25hbWVfX31cIlxuICAgIGFzc2VydCBzZXQoYSkgPT0gc2V0KGIpLCBmXCJ7d2hhdH0ga2V5cyB7c29ydGVkKGEpfSwgZXhwZWN0ZWQge3NvcnRlZChiKX1cIlxuXG5kZWYgX3QoZGYpOlxuICAgIGEgPSBVW1widHJhbnNmb3JtX3IyXCJdKGRmKTsgYiA9IFJbXCJ0cmFuc2Zvcm1fcjJcIl0oZGYpOyBzYW1lX2tleXMoYSwgYilcbiAgICBmb3IgayBpbiBiOiBjbG9zZShhW2tdLCBiW2tdLCAxZS02LCBcIlJcdTAwYjIgZm9yIFwiICsgaylcbiAgICBmb3IgayBpbiAoXCJpbnZlcnNlXCIsIFwibG9nXCIpOlxuICAgICAgICB4LCB5ID0gVVtcInNsb3BlX2NpODBcIl0oZGYsIGspLCBSW1wic2xvcGVfY2k4MFwiXShkZiwgaylcbiAgICAgICAgY2xvc2UoeFswXSwgeVswXSwgMWUtNiwgayArIFwiIENJIGxvd1wiKTsgY2xvc2UoeFsxXSwgeVsxXSwgMWUtNiwgayArIFwiIENJIGhpZ2hcIilcblRFU1RTID0gWyhcImF1dG9cIiwgbGFtYmRhOiBfdChhdXRvKSksIChcImhpZGRlbjogY2FycyBmcm9tIDE5NzYgb25cIiwgbGFtYmRhOiBfdChhdXRvLmZpbHRlcihwbC5jb2woXCJ5ZWFyXCIpID49IDc2KSkpXVxuIiwgImxhbmciOiAicHkifSwgInI0IjogeyJzb2x1dGlvbiI6ICJkZWYgY2F0c19tb2RlbHMoZGYpOlxuICAgIG0gPSBzbWYub2xzKFwiSHd0IH4gU2V4ICogQnd0XCIsIGRhdGE9ZGYudG9fcGFuZGFzKCkpLmZpdCgpXG4gICAgdGVybSA9IFt0IGZvciB0IGluIG0ucGFyYW1zLmluZGV4IGlmIFwiOlwiIGluIHRdWzBdXG4gICAgc2xvcGUgPSBsYW1iZGEgczogZmxvYXQoc21mLm9scyhcIkh3dCB+IEJ3dFwiLCBkYXRhPWRmLmZpbHRlcihwbC5jb2woXCJTZXhcIikgPT0gcykudG9fcGFuZGFzKCkpLmZpdCgpLnBhcmFtc1tcIkJ3dFwiXSlcbiAgICByZXR1cm4ge1wiaW50ZXJhY3Rpb25fcFwiOiBmbG9hdChtLnB2YWx1ZXNbdGVybV0pLCBcImZfcHZhbHVlXCI6IGZsb2F0KG0uZl9wdmFsdWUpLFxuICAgICAgICAgICAgXCJzbG9wZV9GXCI6IHNsb3BlKFwiRlwiKSwgXCJzbG9wZV9NXCI6IHNsb3BlKFwiTVwiKX1cbiIsICJ0ZXN0cyI6ICJcbmRlZiBjbG9zZShhLCBiLCByZWw9MWUtNiwgd2hhdD1cInZhbHVlXCIpOlxuICAgIGFzc2VydCBhIGlzIG5vdCBOb25lLCBcInlvdXIgZnVuY3Rpb24gcmV0dXJuZWQgTm9uZS4gRGlkIHlvdSBmb3JnZXQgdG8gcmV0dXJuP1wiXG4gICAgYSA9IGZsb2F0KGEpOyBiID0gZmxvYXQoYilcbiAgICBhc3NlcnQgYWJzKGEgLSBiKSA8PSByZWwgKiBtYXgoMS4wLCBhYnMoYikpLCBmXCJ7d2hhdH0gPSB7YTouNmd9LCBleHBlY3RlZCB7YjouNmd9XCJcbmRlZiBiZXR3ZWVuKGEsIGxvLCBoaSwgd2hhdD1cInZhbHVlXCIpOlxuICAgIGFzc2VydCBhIGlzIG5vdCBOb25lLCBcInlvdXIgZnVuY3Rpb24gcmV0dXJuZWQgTm9uZS4gRGlkIHlvdSBmb3JnZXQgdG8gcmV0dXJuP1wiXG4gICAgYXNzZXJ0IGxvIDw9IGZsb2F0KGEpIDw9IGhpLCBmXCJ7d2hhdH0gPSB7ZmxvYXQoYSk6LjRmfSwgZXhwZWN0ZWQgYmV0d2VlbiB7bG99IGFuZCB7aGl9XCJcbmRlZiBzYW1lX2tleXMoYSwgYiwgd2hhdD1cInJlc3VsdFwiKTpcbiAgICBhc3NlcnQgaXNpbnN0YW5jZShhLCBkaWN0KSwgZlwie3doYXR9IHNob3VsZCBiZSBhIGRpY3QsIGdvdCB7dHlwZShhKS5fX25hbWVfX31cIlxuICAgIGFzc2VydCBzZXQoYSkgPT0gc2V0KGIpLCBmXCJ7d2hhdH0ga2V5cyB7c29ydGVkKGEpfSwgZXhwZWN0ZWQge3NvcnRlZChiKX1cIlxuXG5kZWYgX3QoZGYpOlxuICAgIGEgPSBVW1wiY2F0c19tb2RlbHNcIl0oZGYpOyBiID0gUltcImNhdHNfbW9kZWxzXCJdKGRmKTsgc2FtZV9rZXlzKGEsIGIpXG4gICAgZm9yIGsgaW4gYjogY2xvc2UoYVtrXSwgYltrXSwgMWUtNiwgaylcblRFU1RTID0gWyhcImNhdHNcIiwgbGFtYmRhOiBfdChjYXRzKSksIChcImhpZGRlbjogY2F0cyB1bmRlciAzLjIga2dcIiwgbGFtYmRhOiBfdChjYXRzLmZpbHRlcihwbC5jb2woXCJCd3RcIikgPCAzLjIpKSldXG4iLCAibGFuZyI6ICJweSJ9LCAicjUiOiB7InNvbHV0aW9uIjogImRlZiBkaWZmX2NpKHNlZWQ9MCk6XG4gICAgcm5nID0gbnAucmFuZG9tLmRlZmF1bHRfcm5nKHNlZWQpXG4gICAgWCA9IHJuZy5ub3JtYWwoc2l6ZT0oMTAwLCAyKSlcbiAgICBlcHMgPSBybmcubm9ybWFsKHNpemU9MTAwKVxuICAgIHkgPSAzICsgNCAqIFhbOiwgMF0gKyA3ICogWFs6LCAxXSArIGVwc1xuICAgIG0gPSBzbS5PTFMoeSwgc20uYWRkX2NvbnN0YW50KFgpKS5maXQoKVxuICAgIGEgPSBucC5hcnJheShbMC4wLCAtMS4wLCAxLjBdKVxuICAgIHBzaSA9IGEgQCBtLnBhcmFtc1xuICAgIHNlID0gbnAuc3FydChhIEAgbS5jb3ZfcGFyYW1zKCkgQCBhKVxuICAgIGggPSBzdGF0cy50LnBwZigwLjk1LCBtLmRmX3Jlc2lkKSAqIHNlXG4gICAgcmV0dXJuIGZsb2F0KHBzaSAtIGgpLCBmbG9hdChwc2kgKyBoKVxuIiwgInRlc3RzIjogIlxuZGVmIGNsb3NlKGEsIGIsIHJlbD0xZS02LCB3aGF0PVwidmFsdWVcIik6XG4gICAgYXNzZXJ0IGEgaXMgbm90IE5vbmUsIFwieW91ciBmdW5jdGlvbiByZXR1cm5lZCBOb25lLiBEaWQgeW91IGZvcmdldCB0byByZXR1cm4/XCJcbiAgICBhID0gZmxvYXQoYSk7IGIgPSBmbG9hdChiKVxuICAgIGFzc2VydCBhYnMoYSAtIGIpIDw9IHJlbCAqIG1heCgxLjAsIGFicyhiKSksIGZcInt3aGF0fSA9IHthOi42Z30sIGV4cGVjdGVkIHtiOi42Z31cIlxuZGVmIGJldHdlZW4oYSwgbG8sIGhpLCB3aGF0PVwidmFsdWVcIik6XG4gICAgYXNzZXJ0IGEgaXMgbm90IE5vbmUsIFwieW91ciBmdW5jdGlvbiByZXR1cm5lZCBOb25lLiBEaWQgeW91IGZvcmdldCB0byByZXR1cm4/XCJcbiAgICBhc3NlcnQgbG8gPD0gZmxvYXQoYSkgPD0gaGksIGZcInt3aGF0fSA9IHtmbG9hdChhKTouNGZ9LCBleHBlY3RlZCBiZXR3ZWVuIHtsb30gYW5kIHtoaX1cIlxuZGVmIHNhbWVfa2V5cyhhLCBiLCB3aGF0PVwicmVzdWx0XCIpOlxuICAgIGFzc2VydCBpc2luc3RhbmNlKGEsIGRpY3QpLCBmXCJ7d2hhdH0gc2hvdWxkIGJlIGEgZGljdCwgZ290IHt0eXBlKGEpLl9fbmFtZV9ffVwiXG4gICAgYXNzZXJ0IHNldChhKSA9PSBzZXQoYiksIGZcInt3aGF0fSBrZXlzIHtzb3J0ZWQoYSl9LCBleHBlY3RlZCB7c29ydGVkKGIpfVwiXG5cbmRlZiBfdChzZWVkKTpcbiAgICBhID0gVVtcImRpZmZfY2lcIl0oc2VlZCk7IGIgPSBSW1wiZGlmZl9jaVwiXShzZWVkKVxuICAgIGNsb3NlKGFbMF0sIGJbMF0sIDFlLTYsIFwiQ0kgbG93XCIpOyBjbG9zZShhWzFdLCBiWzFdLCAxZS02LCBcIkNJIGhpZ2hcIilcblRFU1RTID0gWyhcInNlZWQgMFwiLCBsYW1iZGE6IF90KDApKSwgKFwiaGlkZGVuOiBzZWVkIDExXCIsIGxhbWJkYTogX3QoMTEpKV1cbiIsICJsYW5nIjogInB5In0sICJyNiI6IHsic29sdXRpb24iOiAiZGVmIG91dGxpZXJfZWZmZWN0KHNlZWQ9MCk6XG4gICAgcm5nID0gbnAucmFuZG9tLmRlZmF1bHRfcm5nKHNlZWQpXG4gICAgeCA9IHJuZy5ub3JtYWwoc2l6ZT0yMClcbiAgICBlcHMgPSBybmcubm9ybWFsKHNpemU9MjApXG4gICAgeSA9IDMgKyA1ICogeCArIGVwc1xuICAgIGZpdCA9IGxhbWJkYSB5eTogc20uT0xTKHl5LCBzbS5hZGRfY29uc3RhbnQoeCkpLmZpdCgpXG4gICAgbSA9IGZpdCh5KVxuICAgIHkyID0geS5jb3B5KCk7IHkyWzBdID0gMTAwXG4gICAgbTIgPSBmaXQoeTIpXG4gICAgY2kgPSBsYW1iZGEgbW06IHR1cGxlKGZsb2F0KHYpIGZvciB2IGluIG1tLmNvbmZfaW50KGFscGhhPTAuMDUpWzFdKVxuICAgIHJldHVybiB7XCJzbG9wZVwiOiBmbG9hdChtLnBhcmFtc1sxXSksIFwiY2lcIjogY2kobSksIFwic2xvcGVfb3V0XCI6IGZsb2F0KG0yLnBhcmFtc1sxXSksIFwiY2lfb3V0XCI6IGNpKG0yKX1cbiIsICJ0ZXN0cyI6ICJcbmRlZiBjbG9zZShhLCBiLCByZWw9MWUtNiwgd2hhdD1cInZhbHVlXCIpOlxuICAgIGFzc2VydCBhIGlzIG5vdCBOb25lLCBcInlvdXIgZnVuY3Rpb24gcmV0dXJuZWQgTm9uZS4gRGlkIHlvdSBmb3JnZXQgdG8gcmV0dXJuP1wiXG4gICAgYSA9IGZsb2F0KGEpOyBiID0gZmxvYXQoYilcbiAgICBhc3NlcnQgYWJzKGEgLSBiKSA8PSByZWwgKiBtYXgoMS4wLCBhYnMoYikpLCBmXCJ7d2hhdH0gPSB7YTouNmd9LCBleHBlY3RlZCB7YjouNmd9XCJcbmRlZiBiZXR3ZWVuKGEsIGxvLCBoaSwgd2hhdD1cInZhbHVlXCIpOlxuICAgIGFzc2VydCBhIGlzIG5vdCBOb25lLCBcInlvdXIgZnVuY3Rpb24gcmV0dXJuZWQgTm9uZS4gRGlkIHlvdSBmb3JnZXQgdG8gcmV0dXJuP1wiXG4gICAgYXNzZXJ0IGxvIDw9IGZsb2F0KGEpIDw9IGhpLCBmXCJ7d2hhdH0gPSB7ZmxvYXQoYSk6LjRmfSwgZXhwZWN0ZWQgYmV0d2VlbiB7bG99IGFuZCB7aGl9XCJcbmRlZiBzYW1lX2tleXMoYSwgYiwgd2hhdD1cInJlc3VsdFwiKTpcbiAgICBhc3NlcnQgaXNpbnN0YW5jZShhLCBkaWN0KSwgZlwie3doYXR9IHNob3VsZCBiZSBhIGRpY3QsIGdvdCB7dHlwZShhKS5fX25hbWVfX31cIlxuICAgIGFzc2VydCBzZXQoYSkgPT0gc2V0KGIpLCBmXCJ7d2hhdH0ga2V5cyB7c29ydGVkKGEpfSwgZXhwZWN0ZWQge3NvcnRlZChiKX1cIlxuXG5kZWYgX3Qoc2VlZCk6XG4gICAgYSA9IFVbXCJvdXRsaWVyX2VmZmVjdFwiXShzZWVkKTsgYiA9IFJbXCJvdXRsaWVyX2VmZmVjdFwiXShzZWVkKTsgc2FtZV9rZXlzKGEsIGIpXG4gICAgZm9yIGsgaW4gKFwic2xvcGVcIiwgXCJzbG9wZV9vdXRcIik6IGNsb3NlKGFba10sIGJba10sIDFlLTYsIGspXG4gICAgZm9yIGsgaW4gKFwiY2lcIiwgXCJjaV9vdXRcIik6XG4gICAgICAgIGNsb3NlKGFba11bMF0sIGJba11bMF0sIDFlLTYsIGsgKyBcIiBsb3dcIik7IGNsb3NlKGFba11bMV0sIGJba11bMV0sIDFlLTYsIGsgKyBcIiBoaWdoXCIpXG5URVNUUyA9IFsoXCJzZWVkIDBcIiwgbGFtYmRhOiBfdCgwKSksIChcImhpZGRlbjogc2VlZCA1XCIsIGxhbWJkYTogX3QoNSkpXVxuIiwgImxhbmciOiAicHkifSwgInI3IjogeyJzb2x1dGlvbiI6ICJkZWYgaGMxX3ZzX2NsYXNzaWMoZGYpOlxuICAgIGQgPSBkZi5kcm9wX251bGxzKFtcIk96b25lXCIsIFwiU29sYXIuUlwiXSlcbiAgICBYID0gc20uYWRkX2NvbnN0YW50KGRbXCJTb2xhci5SXCJdLnRvX251bXB5KCkuYXN0eXBlKGZsb2F0KSlcbiAgICB5ID0gZFtcIk96b25lXCJdLnRvX251bXB5KCkuYXN0eXBlKGZsb2F0KVxuICAgIHJldHVybiBmbG9hdChzbS5PTFMoeSwgWCkuZml0KCkuYnNlWzFdKSwgZmxvYXQoc20uT0xTKHksIFgpLmZpdChjb3ZfdHlwZT1cIkhDMVwiKS5ic2VbMV0pXG4iLCAidGVzdHMiOiAiXG5kZWYgY2xvc2UoYSwgYiwgcmVsPTFlLTYsIHdoYXQ9XCJ2YWx1ZVwiKTpcbiAgICBhc3NlcnQgYSBpcyBub3QgTm9uZSwgXCJ5b3VyIGZ1bmN0aW9uIHJldHVybmVkIE5vbmUuIERpZCB5b3UgZm9yZ2V0IHRvIHJldHVybj9cIlxuICAgIGEgPSBmbG9hdChhKTsgYiA9IGZsb2F0KGIpXG4gICAgYXNzZXJ0IGFicyhhIC0gYikgPD0gcmVsICogbWF4KDEuMCwgYWJzKGIpKSwgZlwie3doYXR9ID0ge2E6LjZnfSwgZXhwZWN0ZWQge2I6LjZnfVwiXG5kZWYgYmV0d2VlbihhLCBsbywgaGksIHdoYXQ9XCJ2YWx1ZVwiKTpcbiAgICBhc3NlcnQgYSBpcyBub3QgTm9uZSwgXCJ5b3VyIGZ1bmN0aW9uIHJldHVybmVkIE5vbmUuIERpZCB5b3UgZm9yZ2V0IHRvIHJldHVybj9cIlxuICAgIGFzc2VydCBsbyA8PSBmbG9hdChhKSA8PSBoaSwgZlwie3doYXR9ID0ge2Zsb2F0KGEpOi40Zn0sIGV4cGVjdGVkIGJldHdlZW4ge2xvfSBhbmQge2hpfVwiXG5kZWYgc2FtZV9rZXlzKGEsIGIsIHdoYXQ9XCJyZXN1bHRcIik6XG4gICAgYXNzZXJ0IGlzaW5zdGFuY2UoYSwgZGljdCksIGZcInt3aGF0fSBzaG91bGQgYmUgYSBkaWN0LCBnb3Qge3R5cGUoYSkuX19uYW1lX199XCJcbiAgICBhc3NlcnQgc2V0KGEpID09IHNldChiKSwgZlwie3doYXR9IGtleXMge3NvcnRlZChhKX0sIGV4cGVjdGVkIHtzb3J0ZWQoYil9XCJcblxuZGVmIF90KGRmKTpcbiAgICBhID0gVVtcImhjMV92c19jbGFzc2ljXCJdKGRmKTsgYiA9IFJbXCJoYzFfdnNfY2xhc3NpY1wiXShkZilcbiAgICBjbG9zZShhWzBdLCBiWzBdLCAxZS02LCBcImNsYXNzaWMgU0VcIik7IGNsb3NlKGFbMV0sIGJbMV0sIDFlLTYsIFwiSEMxIFNFXCIpXG5URVNUUyA9IFsoXCJhaXJxdWFsaXR5XCIsIGxhbWJkYTogX3QoYWlycXVhbGl0eSkpLCAoXCJoaWRkZW46IE1heSB0byBKdWx5XCIsIGxhbWJkYTogX3QoYWlycXVhbGl0eS5maWx0ZXIocGwuY29sKFwiTW9udGhcIikgPD0gNykpKV1cbiIsICJsYW5nIjogInB5In19").decode())
_SETUP = _b64.b64decode("aW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwb2xhcnMgYXMgcGwKaW1wb3J0IG1hdHBsb3RsaWIucHlwbG90IGFzIHBsdAppbXBvcnQgc3RhdHNtb2RlbHMuYXBpIGFzIHNtCmltcG9ydCBzdGF0c21vZGVscy5mb3JtdWxhLmFwaSBhcyBzbWYKZnJvbSBzY2lweSBpbXBvcnQgc3RhdHMKCmRlZiByZGF0YShuYW1lLCBwYWNrYWdlKToKICAgICIiIkxvYWQgYW4gUiBkYXRhc2V0ICh2aWEgc3RhdHNtb2RlbHMgLyBSZGF0YXNldHMpIGFzIGEgcG9sYXJzIERhdGFGcmFtZSwgTmFOIOKGkiBudWxsLiIiIgogICAgZGYgPSBwbC5mcm9tX3BhbmRhcyhzbS5kYXRhc2V0cy5nZXRfcmRhdGFzZXQobmFtZSwgcGFja2FnZSwgY2FjaGU9VHJ1ZSkuZGF0YS5yZXNldF9pbmRleChkcm9wPVRydWUpKS5kcm9wKCJyb3duYW1lcyIsIHN0cmljdD1GYWxzZSkKICAgIHJldHVybiBkZi53aXRoX2NvbHVtbnMocGwuY29sKHBsLkZsb2F0NjQpLmZpbGxfbmFuKE5vbmUpKQoKYWlycXVhbGl0eSA9IHJkYXRhKCJhaXJxdWFsaXR5IiwgImRhdGFzZXRzIikKYXV0byA9IHJkYXRhKCJBdXRvIiwgIklTTFIiKQpjYXRzID0gcmRhdGEoImNhdHMiLCAiTUFTUyIp").decode()

def _err(e):
    fr = [f for f in _tb.extract_tb(e.__traceback__) if f.filename.startswith("<ipython") or f.filename == "<string>"]
    return f"{type(e).__name__}: {e}"

def check(ex_id, query=None):
    """Run the hidden tests for one exercise, like Submit in the editor."""
    e = _LAB[ex_id]
    ref = {}
    with _ctx.redirect_stdout(_io.StringIO()):
        exec(_SETUP, ref)
        if e["lang"] == "sql":
            ref["__sql__"] = e["solution"]
        else:
            exec(e["solution"], ref)
    if e["lang"] == "sql":
        if query is None:
            print("Pass your query string: check(\"" + ex_id + "\", q)")
            return
        user = {"__sql__": query}
    else:
        user = dict(globals())
    tns = dict(ref)
    tns["U"], tns["R"] = user, ref
    exec(e["tests"], tns)
    ok, tests = 0, tns["TESTS"]
    for name, fn in tests:
        try:
            fn()
            print("  ✓ " + name)
            ok += 1
        except AssertionError as err:
            print("  ✗ " + name + ": " + (str(err) or "wrong answer"))
        except Exception as err:
            print("  ✗ " + name + ": " + _err(err))
    print(("Accepted" if ok == len(tests) else "Wrong answer") + f" · {ok} / {len(tests)} test cases passed")

def solution(ex_id):
    """Print the reference solution. Try once more first."""
    print(_LAB[ex_id]["solution"])

---
## 1. AoR 2.14 ex. 4: simulate OLS 1,000 times with Normal vs Cauchy errors, then with errors in X
*Medium · id `r1`*

<p>All of Regression, exercise 2.14.4, with the simulations checked.</p>
<p>1. <code>sim_slopes(errors="normal", n=200, reps=1000, seed=0)</code>: in each repetition draw Xᵢ ~ Unif(−3, 3), Yᵢ = 5 − 2Xᵢ + εᵢ with ε standard Normal, or standard Cauchy if <code>errors="cauchy"</code>, fit OLS and keep β̂₁. Return the array of <code>reps</code> slopes.</p>
<p>2. <code>sim_attenuation(n=100, reps=1000, seed=0)</code>: Xᵢ ~ Unif(0, 1), Yᵢ = 5 + 3Xᵢ + εᵢ with ε ~ N(0, 1), but you only see Wᵢ = Xᵢ + δᵢ with δ ~ N(0, 2) (variance 2). Regress Y on W; return the slopes.</p>
<p>Plot histograms. What is E[β̂₁] in each case, and why? (Exercise 2.14.11 is the theory behind part 2.)</p>

**Theory to know cold**
<ul><li>OLS stays unbiased with Cauchy errors (conditional on X the slope is a weighted sum of errors), but its variance is infinite: the histogram has huge outliers.</li><li>Errors in X bias the slope toward 0 by the factor Var(X) / (Var(X) + Var(δ)), here (1/12) / (1/12 + 2).</li></ul>

In [ ]:
def sim_slopes(errors="normal", n=200, reps=1000, seed=0):
    rng = np.random.default_rng(seed)
    # your code here
    pass

def sim_attenuation(n=100, reps=1000, seed=0):
    rng = np.random.default_rng(seed)
    # your code here
    pass

# Sample run
try:
    s = sim_slopes(); c = sim_slopes("cauchy"); w = sim_attenuation()
    print(s.mean(), np.median(c), w.mean())
    fig, ax = plt.subplots(1, 3, figsize=(12, 3))
    ax[0].hist(s, 40); ax[1].hist(np.clip(c, -10, 6), 40); ax[2].hist(w, 40); plt.show()
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("r1")

---
## 2. AoR 2.14 ex. 6: airquality, Ozone on Solar.R: fit, residual plot, CI and t-test for the slope
*Easy · id `r2`*

<p>Exercise 2.14.6 on <code>airquality</code> (polars). Drop rows where <code>Ozone</code> or <code>Solar.R</code> is null.</p>
<p>Write <code>ozone_fit(df)</code> returning a dict: <code>intercept</code>, <code>slope</code>, <code>ci90</code> (a 90% CI for the slope as a tuple), <code>p_value</code> (two-sided t-test of slope = 0). Use statsmodels; get numpy arrays from polars with <code>.to_numpy()</code>.</p>
<p>Then plot the data with the line, and the residuals against Solar.R. Do the model assumptions look right?</p>

**Theory to know cold**
<ul><li>CI for βⱼ: β̂ⱼ ± t_{n−2, 0.95} · se(β̂ⱼ) for 90%.</li><li>A fan-shaped residual plot means heteroscedasticity: the standard errors are wrong, not the slope.</li></ul>

In [ ]:
def ozone_fit(df: pl.DataFrame) -> dict:
    # your code here
    pass

# Sample run
try:
    print(ozone_fit(airquality))
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("r2")

---
## 3. AoR 2.14 ex. 8: Auto, mpg on weight: try √, log and 1/x, read the residual plots
*Medium · id `r3`*

<p>Exercise 2.14.8 with <code>auto</code> (ISLR's version of auto-mpg, polars).</p>
<p>1. <code>transform_r2(df)</code>: regress <code>mpg</code> on weight, √weight, log(weight) and 1/weight. Return a dict with keys <code>"linear", "sqrt", "log", "inverse"</code> → R². Build the transformed columns with polars expressions (<code>pl.col("weight").sqrt()</code>, <code>.log()</code>, <code>1 / pl.col("weight")</code>).</p>
<p>2. <code>slope_ci80(df, kind)</code>: the 80% CI (tuple) for the slope of the chosen transform.</p>
<p>Plot residuals vs fitted values for each. Which transform fixes the curvature, and what is still wrong?</p>

**Theory to know cold**
<ul><li>Curved residuals mean the mean function is wrong; a funnel means non-constant variance. A transform of x fixes the first, a transform of y the second.</li><li>R² only compares models with the same response.</li></ul>

In [ ]:
KINDS = {"linear": pl.col("weight"), "sqrt": pl.col("weight").sqrt(),
         "log": pl.col("weight").log(), "inverse": 1 / pl.col("weight")}

def transform_r2(df: pl.DataFrame) -> dict:
    # your code here
    pass

def slope_ci80(df: pl.DataFrame, kind: str):
    # your code here
    pass

# Sample run
try:
    print(transform_r2(auto))
    print(slope_ci80(auto, "inverse"))
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("r3")

---
## 4. AoR 2.14 ex. 13: cats, a dummy for Sex, the Sex × Bwt interaction and the F-test
*Medium · id `r4`*

<p>Exercise 2.14.13 with <code>cats</code> (Sex F/M, body weight <code>Bwt</code>, heart weight <code>Hwt</code>).</p>
<p>Write <code>cats_models(df)</code> returning a dict:</p><ul>
<li><code>interaction_p</code>: p-value of the Sex × Bwt term in <code>Hwt ~ Sex * Bwt</code>;</li>
<li><code>f_pvalue</code>: the overall F-test p-value of that model;</li>
<li><code>slope_F</code>, <code>slope_M</code>: the Bwt slope from separate regressions on females and males (split with polars <code>filter</code>).</li></ul>
<p>The formula API needs pandas, so convert only at the call: <code>smf.ols("Hwt ~ Sex * Bwt", data=df.to_pandas())</code>.</p>
<p>Plot both fitted lines. Is the interaction significant at 10%? What weaker assumption do the separate regressions make?</p>

**Theory to know cold**
<ul><li>With an interaction, the Bwt coefficient is the female slope and Bwt + Sex:Bwt is the male slope, so the joint model reproduces the separate slopes.</li><li>Separate fits also let the error variance differ by sex.</li></ul>

In [ ]:
def cats_models(df: pl.DataFrame) -> dict:
    # your code here
    pass

# Sample run
try:
    print(cats_models(cats))
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("r4")

---
## 5. AoR 2.14 ex. 15 (b): 90% CI for β₂ − β₁ from the fitted covariance matrix
*Medium · id `r5`*

<p>Exercise 2.14.15. Write <code>diff_ci(seed=0)</code>: with <code>rng = np.random.default_rng(seed)</code> draw <code>X = rng.normal(size=(100, 2))</code> then <code>eps = rng.normal(size=100)</code> (in that order), set Y = 3 + 4X₁ + 7X₂ + ε, fit OLS with a constant, and return the 90% CI for β₂ − β₁ as a tuple.</p>
<p>Build it from <code>model.cov_params()</code> with a = (0, −1, 1): ψ̂ ± t_{n−3, 0.95} √(aᵀΓa). Check it against <code>model.t_test(a)</code>.</p>

**Theory to know cold**
<ul><li>Var(aᵀβ̂) = aᵀ Γ a, with Γ = σ̂²(XᵀX)⁻¹. The covariance between β̂₁ and β̂₂ matters: ignoring it gives the wrong width.</li></ul>

In [ ]:
def diff_ci(seed=0):
    rng = np.random.default_rng(seed)
    # your code here
    pass

# Sample run
try:
    print(diff_ci())
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("r5")

---
## 6. AoR 2.14 ex. 16: add one outlier and compare slope and CI
*Easy · id `r6`*

<p>Exercise 2.14.16. Write <code>outlier_effect(seed=0)</code>: draw <code>x = rng.normal(size=20)</code> then <code>eps = rng.normal(size=20)</code>, Y = 3 + 5x + ε. Fit OLS; then set Y₁ = 100 (the first element) and refit.</p>
<p>Return a dict: <code>slope</code>, <code>ci</code> (95%, tuple), <code>slope_out</code>, <code>ci_out</code>. How much does one point move the slope, and what happens to the interval?</p>

In [ ]:
def outlier_effect(seed=0):
    rng = np.random.default_rng(seed)
    # your code here
    pass

# Sample run
try:
    print(outlier_effect())
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("r6")

---
## 7. Refit any model with cov_type='HC1' and explain why the standard errors moved
*Easy · id `r7`*

<p>Write <code>hc1_vs_classic(df)</code> for the Ozone ~ Solar.R fit on <code>airquality</code> (nulls dropped): return <code>(se_classic, se_hc1)</code>, the slope's standard error from the usual fit and from <code>.fit(cov_type="HC1")</code>.</p>
<p>The slope itself doesn't change. Why does its standard error? Link it to the residual plot from exercise 2.</p>

**Theory to know cold**
<ul><li>Classic SEs assume Var(ε|x) = σ². White / HC SEs use the squared residuals as estimates of each observation's variance (HC1 adds an n/(n−d−1) correction).</li><li>Robust SEs are the default in empirical finance; with time-series data use HAC (Newey-West) instead.</li></ul>

In [ ]:
def hc1_vs_classic(df: pl.DataFrame):
    # your code here
    pass

# Sample run
try:
    print(hc1_vs_classic(airquality))
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("r7")